## Lab 06 – wyrażenia regularne, collections i klasy

Rozgrzewka: kurs wyrażeń regularnych – regexone.com

1. Filtrowanie logu eksperymentu – `successful_runs`
2. Analiza logu systemowego – `invalid_user_lines`, `format_invalid_users`
3. `defaultdict` – `count_letters`, `group_words_by_first_letter`
4. Niemodyfikowalny słownik – `FrozenDictionary`
5. Worek słów – `BagOfWords`
6. Własne `staticmethod`, `classmethod` i `property`

Zadanie dodatkowe: LRU Cache (LeetCode 146)

### Przypomnienie: wyrażenia regularne

* `re.search(wzorzec, tekst)` – szuka **pierwszego** dopasowania w dowolnym miejscu tekstu (zwraca obiekt `Match` albo `None`),
* `re.fullmatch(wzorzec, tekst)` – sprawdza, czy **cały** tekst pasuje do wzorca,
* `re.findall(wzorzec, tekst)` – zwraca listę wszystkich dopasowań,
* `(?P<nazwa>...)` – grupa nazwana; jej zawartość odczytasz przez `m.group("nazwa")` albo `m.groupdict()`.

Wzorce zapisuj jako *raw stringi* (`r"..."`), żeby `\d` czy `\s` nie były interpretowane przez Pythona.

In [ ]:
import re

text = "Zajęcia: 2026-10-27 i 2026-11-03, sala 1.18"

print(re.findall(r"\d{4}-\d{2}-\d{2}", text))

m = re.search(r"(?P<year>\d{4})-(?P<month>\d{2})-(?P<day>\d{2})", text)
print(m.group("day"), m.groupdict())

print(re.fullmatch(r"\d{4}-\d{2}-\d{2}", "2026-10-27") is not None)
print(re.fullmatch(r"\d{4}-\d{2}-\d{2}", "2026-10-27!") is not None)

## Rozgrzewka

Przejdź 15 lekcji kursu [regexone.com](https://regexone.com/). Każda lekcja to krótkie wyjaśnienie i kilka przykładów do dopasowania – przydadzą się w zadaniach 1 i 2.

## Zadanie 1

Plik `data/atoms.log` zawiera log z serii eksperymentów. Każda linia ma postać:

```
RUN [6 CYFR] COMPLETED. OUTPUT IN FILE [NAZWA].dat. [OPCJONALNIE: OPIS PROBLEMU]
```

Interesują nas tylko eksperymenty, które przebiegły bez problemów – czyli linie, w których po `NAZWA.dat.` nie ma już nic. Odrzucone powinny zostać np.:

```
RUN 000106 COMPLETED. OUTPUT IN FILE fluorine.dat. 1 OVERFLOW WARNING.
RUN 000229 COMPLETED. OUTPUT IN FILE sodium.dat. ALGORITHM DID NOT CONVERGE AFTER 100000 ITERATIONS.
```

Napisz funkcję `successful_runs(path)`, która przy pomocy wyrażenia regularnego zwraca listę takich linii (bez znaku końca linii `\n`), w kolejności z pliku.

In [ ]:
import re


def successful_runs(path):
    """Zwraca listę linii z pliku `path` opisujących eksperymenty bez problemów"""
    pass

In [ ]:
runs = successful_runs("data/atoms.log")

print(len(runs) == 32)
print(runs[0] == "RUN 000003 COMPLETED. OUTPUT IN FILE zinc.dat.")
print("RUN 000835 COMPLETED. OUTPUT IN FILE yttrium.dat." in runs)
print("RUN 000106 COMPLETED. OUTPUT IN FILE fluorine.dat. 1 OVERFLOW WARNING." not in runs)
print(not any(line.endswith("\n") for line in runs))

## Zadanie 2

Plik `data/messages.txt` to fragment logu systemowego serwera. Część linii dotyczy prób logowania przez SSH na nieistniejące konto, np.:

```
Jun 29 20:18:40 noether sshd[5805]: Invalid user tester from 218.189.194.200
```

a) Napisz funkcję `invalid_user_lines(path)`, która zwraca listę linii (bez `\n`) w dokładnie takiej postaci – z komunikatem `Invalid user NAZWA from ADRES_IP`. Uwaga: linie typu `Failed password for invalid user ...` to inny komunikat i nie powinny zostać zwrócone.

In [ ]:
import re


def invalid_user_lines(path):
    """Zwraca linie z pliku `path` z komunikatem „Invalid user NAZWA from ADRES_IP”"""
    pass

In [ ]:
lines = invalid_user_lines("data/messages.txt")

print(len(lines) == 6)
print(lines[0] == "Jun 29 20:18:40 noether sshd[5805]: Invalid user tester from 218.189.194.200")
print(not any("Failed password" in line for line in lines))
print(not any(line.endswith("\n") for line in lines))

b) Napisz funkcję `format_invalid_users(path)`, która dla tych samych linii zwraca listę czytelnych komunikatów, np. dla linii powyżej:

```
Niepoprawna nazwa użytkownika: "tester", połączenie z 218.189.194.200 nawiązano 29 czerwca o godz. 20:18
```

Wymagania:
* użyj grup nazwanych (`(?P<nazwa>...)`),
* sekundy pomijamy, dzień piszemy bez zera na początku,
* w logu dni jednocyfrowe są wyrównane spacją, np. `Jul  1 03:14:07` – wzorzec musi to obsłużyć.

Słownik z nazwami miesięcy jest już przygotowany.

In [ ]:
MONTHS = {
    "Jan": "stycznia", "Feb": "lutego", "Mar": "marca", "Apr": "kwietnia",
    "May": "maja", "Jun": "czerwca", "Jul": "lipca", "Aug": "sierpnia",
    "Sep": "września", "Oct": "października", "Nov": "listopada", "Dec": "grudnia",
}


def format_invalid_users(path):
    """Zwraca listę komunikatów o próbach logowania na nieistniejące konta"""
    pass

In [ ]:
messages = format_invalid_users("data/messages.txt")

print(len(messages) == 6)
print(messages[0] == 'Niepoprawna nazwa użytkownika: "tester", połączenie z 218.189.194.200 nawiązano 29 czerwca o godz. 20:18')
print('Niepoprawna nazwa użytkownika: "guest", połączenie z 5.188.10.176 nawiązano 1 lipca o godz. 03:14' in messages)
print(messages[-1] == 'Niepoprawna nazwa użytkownika: "ubnt", połączenie z 185.220.101.4 nawiązano 12 lipca o godz. 04:41')

<details>
<summary><b>Ciekawostka:</b> czytelne wyrażenia regularne – `re.VERBOSE`</summary>

Dłuższe wzorce szybko stają się nieczytelne. Flaga [`re.VERBOSE`](https://docs.python.org/3/library/re.html#re.VERBOSE) pozwala rozbić wzorzec na wiele linii i dodać komentarze – białe znaki we wzorcu są wtedy ignorowane (dosłowną spację trzeba zapisać jako `\ ` albo `\s`).

```python
date = re.compile(r"""
    (?P<year>\d{4})    # rok
    -
    (?P<month>\d{2})   # miesiąc
    -
    (?P<day>\d{2})     # dzień
""", re.VERBOSE)

print(date.fullmatch("2026-10-27").groupdict())
```
```
{'year': '2026', 'month': '10', 'day': '27'}
```

</details>

## Zadanie 3

[`collections.defaultdict`](https://docs.python.org/3/library/collections.html#collections.defaultdict) to słownik, który przy odczycie brakującego klucza sam tworzy dla niego wartość domyślną – np. `defaultdict(int)` zaczyna od `0`, a `defaultdict(list)` od pustej listy. Dzięki temu nie trzeba sprawdzać, czy klucz już istnieje.

Uzupełnij poniższe funkcje, korzystając z `defaultdict`.

In [ ]:
from collections import defaultdict


def count_letters(word):
    """Zwraca słownik {litera: liczba wystąpień w word}"""
    pass


def group_words_by_first_letter(text):
    """Dzieli tekst po spacjach i zwraca słownik {pierwsza litera: lista słów zaczynających się na tę literę}"""
    pass

In [ ]:
print(count_letters("aaaaa") == {"a": 5})
print(count_letters("abbccaaaa") == {"a": 5, "b": 2, "c": 2})
print(count_letters("abc") == {"a": 1, "b": 1, "c": 1})
print(count_letters("") == {})

print(group_words_by_first_letter("ala ma kota") == {"a": ["ala"], "m": ["ma"], "k": ["kota"]})
print(group_words_by_first_letter("ala ma kota ala ma psa") == {"a": ["ala", "ala"], "m": ["ma", "ma"], "k": ["kota"], "p": ["psa"]})
print(group_words_by_first_letter("ala ma kota ale marysia ma konia") == {"a": ["ala", "ale"], "m": ["ma", "marysia", "ma"], "k": ["kota", "konia"]})
print(group_words_by_first_letter("") == {})

<details>
<summary><b>Ciekawostka:</b> jak działa `defaultdict` – metoda `__missing__`</summary>

Gdy w słowniku brakuje klucza, `dict.__getitem__` zanim rzuci `KeyError`, sprawdza, czy klasa ma metodę `__missing__` – i jeśli tak, zwraca jej wynik. Właśnie tak zbudowany jest `defaultdict`. Własna podklasa `dict` z `__missing__` może przy tym zachowywać się inaczej, np. zwracać wartość domyślną **bez** dodawania klucza:

```python
class ZeroDefault(dict):
    def __missing__(self, key):
        return 0


d = ZeroDefault(ala=3)
print(d["ala"], d["pies"])
print("pies" in d)

from collections import defaultdict

dd = defaultdict(int, ala=3)
print(dd["pies"])
print("pies" in dd)
```
```
3 0
False
0
True
```

`defaultdict` przy odczycie brakującego klucza **dodaje** go do słownika – o tym trzeba pamiętać w zadaniu 5, gdzie odczyt nieznanego słowa nie może zmieniać worka.

*Więcej: Fluent Python (2. wyd.), rozdz. 3.*

</details>

## Zadanie 4

Python ma `frozenset` – niemodyfikowalny odpowiednik `set`, który (w przeciwieństwie do `set`) może być elementem zbioru albo kluczem słownika. Odpowiednika dla słowników nie ma, więc go napiszemy.

Uzupełnij klasę `FrozenDictionary`:
* `__init__(self, dictionary)` – zapamiętuje **kopię** danych ze słownika (późniejsza zmiana oryginału nie wpływa na nasz obiekt),
* `__getitem__(self, key)` – odczyt wartości: `fd["ala"]`,
* `__eq__(self, other)` – dwa obiekty są równe, gdy mają te same pary klucz–wartość (kolejność nie ma znaczenia),
* `__hash__(self)` – zwraca `int`; równe obiekty muszą mieć równe hasze,
* `__repr__(self)` – zwraca np. `FrozenDictionary({'ala': 4})`.

Nie definiuj `__setitem__` – wtedy przypisanie `fd["ala"] = 1` samo rzuci `TypeError`.

<details>
<summary>Podpowiedź</summary>

Hasz musi być taki sam niezależnie od kolejności par. Przyda się `frozenset` – zbiór par `(klucz, wartość)` jest hashowalny, o ile wartości są hashowalne.

</details>

In [ ]:
class FrozenDictionary:
    """Niemodyfikowalny słownik, który może być elementem zbioru albo kluczem słownika"""

    def __init__(self, dictionary):
        pass

    def __getitem__(self, key):
        pass

    def __eq__(self, other):
        pass

    def __hash__(self):
        pass

    def __repr__(self):
        pass

In [ ]:
dicts = [
    FrozenDictionary({"ala": 4}),
    FrozenDictionary({"ala": 1, "jacek": 0}),
    FrozenDictionary({"ala": 4}),
    FrozenDictionary({"ala": 2}),
    FrozenDictionary({"jacek": 0, "ala": 1}),
]

print(dicts[0] == dicts[2])
print(dicts[1] == dicts[4])  # kolejność kluczy nie ma znaczenia
print(dicts[0] != dicts[3])
print(len(set(dicts)) == 3)
print({dicts[0]: "klucz"}[dicts[2]] == "klucz")  # może być kluczem słownika
print(dicts[1]["jacek"] == 0)
print(repr(dicts[0]) == "FrozenDictionary({'ala': 4})")

original = {"ala": 1}
fd = FrozenDictionary(original)
original["ala"] = 99
print(fd["ala"] == 1)  # zmiana oryginału nie wpływa na kopię

try:
    dicts[0]["ola"] = 1
    print(False)
except TypeError:
    print(True)

> **Do przemyślenia:** dlaczego zwykły `dict` nie może być elementem zbioru? I dlaczego równe obiekty muszą mieć równe hasze?

<details>
<summary>Odpowiedź</summary>

Zbiór (i słownik) znajduje element po haszu – na jego podstawie wybiera miejsce w pamięci, a dopiero potem porównuje elementy przez `==`. Gdyby `dict` był hashowalny, to po jego modyfikacji hasz by się zmienił, a zbiór szukałby go w złym miejscu – element „zgubiłby się” w zbiorze. Dlatego obiekty modyfikowalne są niehashowalne, a `FrozenDictionary` musi być niemodyfikowalny.

Gdyby dwa równe obiekty miały różne hasze, zbiór w ogóle nie porównałby ich przez `==` i trzymałby oba – `len(set(dicts))` wyszłoby 4 zamiast 3. Stąd zasada: `a == b` ⇒ `hash(a) == hash(b)`. Python pilnuje jej częściowo sam: jeśli w klasie zdefiniujesz `__eq__` bez `__hash__`, obiekty stają się niehashowalne.

</details>

<details>
<summary><b>Ciekawostka:</b> `types.MappingProxyType` – słownik tylko do odczytu</summary>

Biblioteka standardowa ma [`MappingProxyType`](https://docs.python.org/3/library/types.html#types.MappingProxyType) – widok na słownik, przez który nie da się niczego zmienić. To jednak tylko *widok*: zmiany oryginału są w nim widoczne, a sam widok nie jest hashowalny – nie zastąpi więc `FrozenDictionary`.

```python
from types import MappingProxyType

d = {"ala": 4}
view = MappingProxyType(d)
print(view["ala"])

try:
    view["ola"] = 1
except TypeError as e:
    print("TypeError:", e)

d["ola"] = 1
print(view)

try:
    hash(view)
except TypeError as e:
    print("TypeError:", e)
```
```
4
TypeError: 'mappingproxy' object does not support item assignment
{'ala': 4, 'ola': 1}
TypeError: unhashable type: 'mappingproxy'
```

*Więcej: Fluent Python (2. wyd.), rozdz. 3.*

</details>

## Zadanie 5

Napisz klasę `BagOfWords` – „worek słów”, który pamięta, ile razy każde słowo wystąpiło w tekście. Słowa to fragmenty rozdzielone białymi znakami (wielkość liter ma znaczenie). Klasa ma udostępniać:

* tworzenie z napisu: `BagOfWords("ala ma kota ala ma ala")` albo z pliku (dowolnego obiektu, po którym iteruje się liniami): `BagOfWords(open("plik.txt"))`,
* `str(bow)` / `print(bow)` w postaci `ala:3, ma:2, kota:1` – od najczęstszego słowa, a przy równej liczbie wystąpień alfabetycznie,
* sprawdzanie `"ala" in bow`,
* iterację `for word in bow:` – słowa w tej samej kolejności co w `str`,
* dodawanie `bow1 + bow2` – zwraca **nowy** worek, nie zmieniając `bow1` ani `bow2`,
* odczyt `bow["ala"]` – liczba wystąpień; dla nieznanego słowa `0` (i odczyt nie dodaje go do worka),
* zmianę `bow["tomek"] = 10`.

In [ ]:
class BagOfWords:
    """Worek słów: pamięta, ile razy każde słowo wystąpiło w tekście"""

    def __init__(self, source):
        pass

In [ ]:
from io import StringIO  # StringIO udaje otwarty plik tekstowy

bow = BagOfWords("ala ma kota ala ma ala")
print(str(bow) == "ala:3, ma:2, kota:1")
print(list(bow) == ["ala", "ma", "kota"])
print("ala" in bow and "pies" not in bow)

print(str(BagOfWords(StringIO("ala ma\nkota ala\n"))) == "ala:2, kota:1, ma:1")

bow1 = BagOfWords("ala ma kota ala ma ala")
bow2 = BagOfWords("tomek tez ma kota")
bow3 = bow1 + bow2
print("tomek" not in bow1 and "tomek" in bow3)
print(str(bow3) == "ala:3, ma:3, kota:2, tez:1, tomek:1")
print(str(bow1) == "ala:3, ma:2, kota:1")  # bow1 bez zmian

print(bow1["ala"] == 3 and bow3["ala"] == 3)
print(bow1["pies"] == 0 and "pies" not in bow1)  # odczyt nie dodaje słowa

bow3["tomek"] = 10
print(list(bow3)[0] == "tomek")

> **Do przemyślenia:** dlaczego `bow1 + bow2` powinno zwracać nowy worek, zamiast dopisywać słowa do `bow1`?

<details>
<summary>Odpowiedź</summary>

Tak działa `+` wszędzie w Pythonie: `a + b` dla liczb, napisów czy list zwraca nowy obiekt i nie zmienia składników. Gdyby `bow1 + bow2` po cichu zmieniało `bow1`, kod typu `wszystko = bow1 + bow2` miałby zaskakujący efekt uboczny. Do modyfikacji „w miejscu” służy osobna metoda `__iadd__`, wywoływana przez `bow1 += bow2`.

</details>

<details>
<summary><b>Ciekawostka:</b> `collections.Counter` to prawie gotowy worek słów</summary>

[`Counter`](https://docs.python.org/3/library/collections.html#collections.Counter) z biblioteki standardowej umie większość tego, co pisaliśmy: zlicza elementy, dodaje się przez `+`, zwraca `0` dla nieznanego klucza (bez dodawania go) i sortuje od najczęstszego. Remisy zostawia jednak w kolejności wstawienia, a nie alfabetycznie.

```python
from collections import Counter

bow1 = Counter("ala ma kota ala ma ala".split())
bow2 = Counter("tomek tez ma kota".split())
bow3 = bow1 + bow2

print(bow3.most_common())
print(bow3["nieznane"], "nieznane" in bow3)
```
```
[('ala', 3), ('ma', 3), ('kota', 2), ('tomek', 1), ('tez', 1)]
0 False
```

</details>

## Zadanie 6

Na wykładzie poznaliśmy `@staticmethod`, `@classmethod` i `@property`. To nie są słowa kluczowe, tylko zwykłe klasy oparte na **protokole deskryptorów**: jeśli atrybut klasy ma metodę `__get__(self, instance, owner)`, to odczyt tego atrybutu zwraca wynik `__get__`. `instance` to obiekt, przez który czytamy atrybut (albo `None` przy odczycie przez klasę), a `owner` – klasa.

Zwykłe funkcje też są deskryptorami – tak powstaje `self`: `a.method` to `method.__get__(a, A)`, czyli funkcja z „przyklejonym” pierwszym argumentem (*bound method*). Uruchom komórkę poniżej.

In [ ]:
class Verbose:
    def __get__(self, instance, owner):
        print(f"__get__(instance={instance!r}, owner={owner.__name__})")
        return 42


class A:
    x = Verbose()

    def method(self, x):
        return x * 2


a = A()
print(A.x)                                     # odczyt przez klasę: instance=None
print(a.x)                                     # odczyt przez instancję
print(A.__dict__["method"].__get__(a, A)(21))  # to samo co a.method(21)

a) Napisz klasę `my_staticmethod`, która działa jak wbudowane `staticmethod`: metoda nie dostaje ani instancji, ani klasy jako pierwszego argumentu.

In [ ]:
class my_staticmethod:
    def __init__(self, func):
        pass

    def __get__(self, instance, owner=None):
        pass

In [ ]:
class MathUtils:
    @my_staticmethod
    def add(a, b):
        return a + b


print(MathUtils.add(2, 3) == 5)
print(MathUtils().add(2, 3) == 5)  # bez dekoratora tu poleciałby TypeError

b) Napisz klasę `my_classmethod`, która działa jak wbudowane `classmethod`: metoda dostaje klasę jako pierwszy argument – również wtedy, gdy wywołamy ją przez instancję albo przez podklasę.

Wskazówka: przyda się [`types.MethodType`](https://docs.python.org/3/library/types.html#types.MethodType) albo `functools.partial`.

In [ ]:
class my_classmethod:
    def __init__(self, func):
        pass

    def __get__(self, instance, owner=None):
        pass

In [ ]:
class Date:
    def __init__(self, year, month, day):
        self.year, self.month, self.day = year, month, day

    @my_classmethod
    def from_string(cls, text):
        """Alternatywny konstruktor: Date.from_string("2026-10-27")"""
        return cls(*map(int, text.split("-")))


class MyDate(Date):
    pass


d = Date.from_string("2026-10-27")
print(isinstance(d, Date) and (d.year, d.month, d.day) == (2026, 10, 27))
print(type(MyDate.from_string("2026-10-27")) is MyDate)          # cls to podklasa
print(type(Date(2000, 1, 1).from_string("2026-10-27")) is Date)  # wywołanie przez instancję

c) Napisz klasę `my_property`, która działa jak wbudowane `property` (na razie tylko do odczytu):
* odczyt przez instancję wywołuje udekorowaną funkcję,
* odczyt przez klasę zwraca sam obiekt `my_property`,
* próba przypisania rzuca `AttributeError`.

Wskazówka: do obsługi przypisania potrzebna jest metoda `__set__(self, instance, value)`.

In [ ]:
class my_property:
    def __init__(self, fget):
        pass

    def __get__(self, instance, owner=None):
        pass

    def __set__(self, instance, value):
        pass

In [ ]:
import math


class Circle:
    def __init__(self, r):
        self.r = r

    @my_property
    def area(self):
        return math.pi * self.r ** 2


c = Circle(2)
print(math.isclose(c.area, 4 * math.pi))
c.r = 3
print(math.isclose(c.area, 9 * math.pi))  # wartość liczona przy każdym odczycie
print(isinstance(Circle.area, my_property))

try:
    c.area = 10
    print(False)
except AttributeError:
    print(True)

> **Do przemyślenia:** dlaczego `c.area = 10` w punkcie c) trafia do `__set__`, zamiast po prostu zapisać `10` w atrybutach obiektu `c`?

<details>
<summary>Odpowiedź</summary>

Deskryptor, który ma `__set__` (tzw. *data descriptor*), ma pierwszeństwo przed słownikiem atrybutów instancji (`c.__dict__`) – zarówno przy przypisaniu, jak i przy odczycie. Dlatego `property` może zablokować przypisanie albo przepuścić je przez setter.

Deskryptor bez `__set__` (np. zwykła funkcja albo `my_staticmethod`) takiego pierwszeństwa nie ma: `obj.method = 5` zapisze `5` w `obj.__dict__` i od tej pory `obj.method` zwróci `5` – metoda zostanie „przykryta” dla tej jednej instancji.

</details>

<details>
<summary><b>Ciekawostka:</b> `__set_name__` – deskryptor zna nazwę swojego atrybutu</summary>

Od Pythona 3.6 deskryptor może mieć metodę `__set_name__(self, owner, name)`, którą Python wywołuje przy tworzeniu klasy i podaje w niej nazwę atrybutu. Dzięki temu jeden deskryptor do walidacji można użyć dla wielu pól:

```python
class Positive:
    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        return instance.__dict__[self.name]

    def __set__(self, instance, value):
        if value <= 0:
            raise ValueError(f"{self.name} musi być dodatnie, a jest {value}")
        instance.__dict__[self.name] = value


class Product:
    price = Positive()
    quantity = Positive()

    def __init__(self, price, quantity):
        self.price = price
        self.quantity = quantity


p = Product(10, 3)
print(p.price, p.quantity)

try:
    Product(10, -1)
except ValueError as e:
    print("ValueError:", e)
```
```
10 3
ValueError: quantity musi być dodatnie, a jest -1
```

Wartość można trzymać w `instance.__dict__` pod tą samą nazwą właśnie dlatego, że deskryptor z `__set__` ma pierwszeństwo przed atrybutami instancji.

*Więcej: Fluent Python (2. wyd.), rozdz. 23.*

</details>

## Zadanie dodatkowe

To zadanie można zrobić **zamiast jednego z zadań 1–6**. Pochodzi z LeetCode: [LRU Cache (146)](https://leetcode.com/problems/lru-cache/).

*LRU cache* (*least recently used*) to pamięć podręczna o ograniczonej pojemności: gdy brakuje miejsca, usuwa element, którego **najdawniej używano**. Napisz klasę `LRUCache`:
* `LRUCache(capacity)` – tworzy pusty cache o pojemności `capacity`,
* `get(key)` – zwraca wartość dla klucza albo `-1`, jeśli go nie ma; odczyt liczy się jako użycie klucza,
* `put(key, value)` – zapisuje wartość (lub aktualizuje istniejącą – to też jest użycie); jeśli po dodaniu nowego klucza przekroczono pojemność, usuwa najdawniej używany klucz.

Obie operacje mają działać w czasie stałym – ostatni test wykonuje 200 000 operacji i rozwiązanie, które przy każdej operacji przegląda wszystkie klucze, się go nie doczeka.

<details>
<summary>Podpowiedź 1</summary>

Do szybkiego wyszukiwania po kluczu potrzebny jest słownik. Trzeba jednak dodatkowo pamiętać **kolejność użycia** kluczy – i umieć w czasie stałym przesunąć klucz na koniec tej kolejki oraz usunąć klucz z jej początku.

</details>

<details>
<summary>Podpowiedź 2</summary>

Obie rzeczy naraz daje [`collections.OrderedDict`](https://docs.python.org/3/library/collections.html#collections.OrderedDict): słownik pamiętający kolejność, z metodami `move_to_end(key)` i `popitem(last=False)`.

</details>

In [ ]:
class LRUCache:
    """Pamięć podręczna o ograniczonej pojemności, usuwająca najdawniej używany klucz"""

    def __init__(self, capacity):
        pass

    def get(self, key):
        pass

    def put(self, key, value):
        pass

In [ ]:
cache = LRUCache(2)
cache.put(1, 1)
cache.put(2, 2)
print(cache.get(1) == 1)
cache.put(3, 3)           # usuwa klucz 2 – najdawniej używany
print(cache.get(2) == -1)
cache.put(4, 4)           # usuwa klucz 1
print(cache.get(1) == -1)
print(cache.get(3) == 3 and cache.get(4) == 4)

cache = LRUCache(2)
cache.put(1, 1)
cache.put(2, 2)
cache.put(1, 10)          # aktualizacja też jest użyciem
cache.put(3, 3)           # więc usuwany jest klucz 2
print(cache.get(1) == 10 and cache.get(2) == -1)

cache = LRUCache(1)
cache.put(1, 1)
cache.put(2, 2)
print(cache.get(1) == -1 and cache.get(2) == 2)

cache = LRUCache(50_000)
for i in range(100_000):
    cache.put(i, i)
print(all(cache.get(i) == -1 for i in range(50_000)) and all(cache.get(i) == i for i in range(50_000, 100_000)))